# 01 Tensor 与 Autograd 基础

## 本节目标

本节关注 PyTorch 中最基础、也最容易被忽略的部分：Tensor 和自动求导。

完成本节后，你应该能够：

- 创建和检查 Tensor，理解 shape、dtype、device 的含义。
- 判断常见张量运算后的形状变化。
- 解释 `requires_grad`、`backward()`、`.grad` 之间的关系。
- 理解为什么训练循环中要清空梯度。
- 用一个最小例子完成“预测 -> loss -> 梯度 -> 参数更新”闭环。

## 背景问题

深度学习训练本质上是在不断调整参数，使模型输出更接近目标。这里有两个基础问题：

1. 数据、参数和中间计算结果用什么表示？
2. 参数应该往哪个方向调整？

在 PyTorch 中，第一个问题主要由 Tensor 解决，第二个问题主要由 Autograd 解决。这个概念可以从代码角度理解为：Tensor 负责参与计算，Autograd 负责记录计算关系并计算梯度。

## 核心概念

- **Tensor**：多维数组，是 PyTorch 中表示数据和参数的基本对象。
- **Shape**：张量每个维度的大小。例如 `[batch, features]` 表示一批样本和每个样本的特征。
- **dtype**：数据类型，例如 `float32`、`int64`。分类标签常用 `torch.long`。
- **device**：张量所在设备，例如 CPU 或 CUDA。
- **Autograd**：自动求导系统，会跟踪可求导运算并计算梯度。
- **计算图**：由 Tensor 运算构成的依赖关系。反向传播会沿计算图计算梯度。

初学者容易在这里混淆“Tensor 能计算”和“Tensor 会求梯度”。只有参与可求导计算、且相关 Tensor 开启 `requires_grad=True` 时，PyTorch 才会记录梯度。

## 最小代码示例：创建 Tensor 与检查属性

调试时可以优先打印 `shape`、`dtype` 和 `device`。这三个信息通常能定位一半以上的初学错误。

In [ ]:
import torch
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
y = torch.randn(2, 3)
labels = torch.tensor([0, 1, 1], dtype=torch.long)

print("x =", x)
print("x shape:", x.shape)
print("x dtype:", x.dtype)
print("x device:", x.device)
print("y shape:", y.shape)
print("labels dtype:", labels.dtype)

## 最小代码示例：Shape、view 与广播

很多 PyTorch 报错都不是算法问题，而是 shape 没有对齐。下面观察几种常见形状变化。

In [ ]:
a = torch.arange(12).float()
print("original:", a.shape)

b = a.view(3, 4)
print("view(3, 4):", b.shape)

c = b.view(3, 1, 4)
print("view(3, 1, 4):", c.shape)

row_bias = torch.tensor([10.0, 20.0, 30.0, 40.0])
print("b + row_bias:")
print(b + row_bias)  # row_bias 会按行广播

## 最小代码示例：矩阵乘法

神经网络中的线性层本质上离不开矩阵乘法。若 `x` 的形状是 `[batch, input_dim]`，权重 `w` 的形状是 `[input_dim, output_dim]`，那么输出形状就是 `[batch, output_dim]`。

In [ ]:
batch_size = 5
input_dim = 4
output_dim = 2

x = torch.randn(batch_size, input_dim)
w = torch.randn(input_dim, output_dim)
b = torch.randn(output_dim)

out = x @ w + b
print("x:", x.shape)
print("w:", w.shape)
print("b:", b.shape)
print("out:", out.shape)

## 最小代码示例：Autograd

这里的关键不是公式本身，而是看清 PyTorch 如何把运算结果和梯度联系起来。对函数 `y = x^2 + 3x`，当 `x=2` 时，导数是 `2x + 3 = 7`。

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x

y.backward()

print("y:", y.item())
print("dy/dx:", x.grad.item())

## 最小代码示例：梯度会累积

PyTorch 默认会累积梯度。这个设计对梯度累积训练有用，但普通训练循环里通常需要每一步清零。

In [ ]:
x = torch.tensor(1.0, requires_grad=True)

y1 = x * 2
y1.backward()
print("after first backward:", x.grad.item())

y2 = x * 3
y2.backward()
print("after second backward:", x.grad.item())

x.grad.zero_()
print("after zero_:", x.grad.item())

## 完整实验：手写一个最小优化过程

这个实验用于观察：只要能定义 loss，Autograd 就能给出参数更新方向。

任务：找到一个参数 `w`，让 `4 * w` 尽量接近目标值 `6`。理论上，最合适的 `w` 接近 `1.5`。

In [ ]:
w = torch.tensor(0.5, requires_grad=True)
target = torch.tensor(6.0)
lr = 0.1
history = []

for step in range(10):
    prediction = 4 * w
    loss = (prediction - target) ** 2
    loss.backward()

    with torch.no_grad():
        w -= lr * w.grad

    history.append((step, w.item(), loss.item(), w.grad.item()))
    w.grad.zero_()

for step, value, loss_value, grad in history:
    print(f"step={step:02d}  w={value:.4f}  loss={loss_value:.4f}  grad={grad:.4f}")

## 实验观察

从运行结果可以看到，`w` 会逐步靠近 1.5，loss 也快速下降。这个实验虽然很小，但已经包含训练循环的核心结构：

```text
前向计算 -> 计算 loss -> backward -> no_grad 中更新参数 -> 清空梯度
```

真实神经网络训练只是把单个参数换成很多参数，把手动更新换成优化器。

## 常见错误

1. **忘记 `requires_grad=True`**  
   现象：反向传播后没有梯度。  
   检查：确认需要训练的 Tensor 或模型参数会被 Autograd 跟踪。

2. **忘记清空梯度**  
   现象：loss 变化异常，训练不稳定。  
   检查：普通训练循环中每个 batch 前调用 `optimizer.zero_grad()`。

3. **在训练阶段误用 `torch.no_grad()`**  
   现象：loss 不能反向传播。  
   检查：训练时的 forward 和 loss 计算不要放进 `no_grad()`。

4. **shape 靠广播“混过去”**  
   现象：代码能跑但结果不对。  
   检查：打印预测值和目标值 shape，确认它们语义一致。

## 概念问答

**Q：Tensor 和 NumPy array 有什么区别？**  
A：二者都能表示多维数组。Tensor 支持自动求导和 GPU 计算，更适合深度学习训练。

**Q：`.grad` 为什么有时是 `None`？**  
A：可能是 Tensor 没有开启 `requires_grad`，也可能它不是叶子节点，或者相关计算没有连接到 loss。

**Q：为什么更新参数时要用 `torch.no_grad()`？**  
A：参数更新本身不是模型计算的一部分，不需要被记录到计算图中。

**Q：为什么 loss 通常要是标量再 backward？**  
A：标量 loss 对参数求导最常见、最明确。如果是非标量，需要额外传入梯度权重。

## 小结

Tensor 解决“数据如何表示”，Autograd 解决“梯度如何计算”。后续所有模型训练都建立在这两个基础上。建议学习后续章节时继续保持一个习惯：遇到问题先打印 shape、dtype、loss 和梯度。